In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.metrics import F1Score

from google.colab import drive
drive.mount('/content/drive') 

In [ ]:
IMG_SIZE = 512
BATCH_SIZE = 8
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/brain-tumor-detection/Epic and CSCR hospital Dataset/'
checkpoint_path = os.path.join(BASE_PATH, 'tumor_cnn_f1_score_hospital_model_512.keras')
log_path = os.path.join(BASE_PATH, 'training_log.csv')

In [ ]:
datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.2,
    rotation_range=15,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory(
    os.path.join(BASE_PATH, 'Train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    os.path.join(BASE_PATH, 'Train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
if os.path.exists(checkpoint_path):
    print(" Resuming training from last best checkpoint...")
    model = load_model(checkpoint_path)
else:
    print(" Building new EfficientNetB0 model for 512x512...")
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = True 

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.4)(x)
    output = Dense(4, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=output)

    f1_metric = F1Score(average='macro', name='f1_score')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy',f1_metric]
    )

callbacks = [
    ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_f1_score', mode='max'),
    CSVLogger(log_path, append=True),
    EarlyStopping(monitor='val_f1_score', patience=7, restore_best_weights=True, mode='max')
]

print(" Starting training on T4 GPU...")
model.fit(train_gen, validation_data=val_gen, epochs=50, callbacks=callbacks)
